# AND Library Integration MWE

Dieses Notebook ist **nicht** das normale AND-MWE, sondern ein getrenntes **Integrationsbeispiel**.

Es zeigt den Fall, in dem ein uebergeordnetes Package das AND-Package **als Python-Library** aufruft und die Anzeige ueber `progress_handler` selbst rendert.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT = Path('/home/ubuntu/Author_Name_Disambiguation')
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from author_name_disambiguation import ProgressEvent, disambiguate_sources

PUBLICATIONS_PATH = REPO_ROOT / 'data/raw/ads/ads_full_run_20260305/publ_final.parquet'
REFERENCES_PATH = REPO_ROOT / 'data/raw/ads/ads_full_run_20260305/refs_final.parquet'
OUTPUT_DIR = REPO_ROOT / 'artifacts/exports/notebook_infer_integration_mwe'
FORCE = True

PUBLICATIONS_PATH, REFERENCES_PATH, OUTPUT_DIR

(PosixPath('/home/ubuntu/Author_Name_Disambiguation/data/raw/ads/ads_full_run_20260305/publ_final.parquet'),
 PosixPath('/home/ubuntu/Author_Name_Disambiguation/data/raw/ads/ads_full_run_20260305/refs_final.parquet'),
 PosixPath('/home/ubuntu/Author_Name_Disambiguation/artifacts/exports/notebook_infer_integration_mwe'))

In [2]:
STAGE_TITLES = {
    'bootstrap': 'bootstrap',
    'load_inputs': 'load inputs',
    'preflight': 'preflight',
    'name_embeddings': 'name embeddings',
    'text_embeddings': 'text embeddings',
    'pair_inference': 'pair inference',
    'clustering': 'clustering',
    'export': 'export',
}

DYNAMIC_STAGES = {'name_embeddings', 'text_embeddings', 'pair_inference', 'clustering'}


class NotebookANDProgressHandler:
    def __init__(self, parent_prefix: str = '[5/12]', parent_label: str = 'Author Disambiguation') -> None:
        self.parent_prefix = parent_prefix
        self.parent_label = parent_label
        self.events: list[ProgressEvent] = []
        self._bars: dict[str, tqdm] = {}
        self._bar_positions = {
            'name_embeddings': 0,
            'text_embeddings': 1,
            'pair_inference': 2,
            'clustering': 3,
        }
        self._printed_parent = False

    def _ensure_parent_header(self) -> None:
        if self._printed_parent:
            return
        tqdm.write(f'{self.parent_prefix} {self.parent_label}')
        self._printed_parent = True

    def _write(self, label: str, message: str) -> None:
        self._ensure_parent_header()
        tqdm.write(f'  {label:<16} {message}')

    def _get_bar(self, event: ProgressEvent) -> tqdm:
        assert event.stage_key is not None
        bar = self._bars.get(event.stage_key)
        if bar is not None:
            return bar
        desc = f'  {STAGE_TITLES.get(event.stage_key, event.stage_key):<16}'
        bar = tqdm(
            total=max(0, int(event.total or 0)),
            desc=desc,
            unit=str(event.unit or 'it'),
            dynamic_ncols=True,
            leave=True,
            position=self._bar_positions.get(event.stage_key, 0),
        )
        self._bars[event.stage_key] = bar
        return bar

    def close(self) -> None:
        for bar in self._bars.values():
            try:
                bar.close()
            except Exception:
                pass

    def __call__(self, event: ProgressEvent) -> None:
        self.events.append(event)
        stage_key = event.stage_key or ''
        label = STAGE_TITLES.get(stage_key, stage_key or 'run')

        if event.kind == 'stage_start' and stage_key not in DYNAMIC_STAGES:
            self._write(label, 'started')
            return

        if event.kind == 'stage_info' and event.message:
            self._write(label, str(event.message))
            return

        if event.kind == 'stage_warning' and event.message:
            self._write(label, f'WARN {event.message}')
            return

        if event.kind == 'stage_progress' and stage_key in DYNAMIC_STAGES:
            bar = self._get_bar(event)
            total = max(0, int(event.total or 0))
            if total > 0 and bar.total != total:
                bar.total = total
            current = max(0, int(event.current or 0))
            delta = current - int(bar.n)
            if delta > 0:
                bar.update(delta)
            else:
                bar.refresh()
            return

        if event.kind == 'stage_done':
            if stage_key in DYNAMIC_STAGES:
                bar = self._bars.get(stage_key)
                if bar is not None and bar.total is not None and bar.total > 0 and bar.n < bar.total:
                    bar.update(int(bar.total - bar.n))
                    bar.refresh()
            if event.message:
                self._write(label, str(event.message))
            return

        if event.kind == 'run_done':
            payload = dict(event.payload or {})
            counts = dict(payload.get('counts') or {})
            self._write(
                'done',
                f"go={payload.get('go')} | publications={counts.get('publications')} | "
                f"references={counts.get('references')} | mentions={counts.get('mentions')} | "
                f"clusters={counts.get('clusters')}"
            )
            if payload.get('summary_path'):
                self._write('summary', str(payload['summary_path']))
            return

        if event.kind == 'run_failed' and event.message:
            self._write('failed', str(event.message))


handler = NotebookANDProgressHandler()
handler

In [3]:
result = disambiguate_sources(
    publications_path=PUBLICATIONS_PATH,
    references_path=REFERENCES_PATH,
    output_dir=OUTPUT_DIR,
    runtime='auto',
    progress=False,
    progress_handler=handler,
    force=FORCE,
)
handler.close()


[5/12] Author Disambiguation
  bootstrap        started
  bootstrap        dataset=notebook_infer_integration_mwe | infer_stage=full | output_root=/home/ubuntu/Author_Name_Disambiguation/artifacts/exports/notebook_infer_integration_mwe
  bootstrap        device=cuda -> cuda | gpu=NVIDIA A100-SXM4-80GB | precision=fp32 | torch=2.10.0+cu126
  bootstrap        run_id=infer_sources_20260402T155125Z_b68359a5 | uid_scope=dataset | references_present=True
  bootstrap        Bootstrap in 0.7s
  load inputs      started
  load inputs      publications=/home/ubuntu/Author_Name_Disambiguation/data/raw/ads/ads_full_run_20260305/publ_final.parquet | references=/home/ubuntu/Author_Name_Disambiguation/data/raw/ads/ads_full_run_20260305/refs_final.parquet
  load inputs      Loaded publications=183,516 references=441,957 canonical_records=547,517 in 38.4s
  preflight        started
  preflight        precomputed_embedding column present=yes
  preflight        specter_sources=547,517 | specter_recompute

I0000 00:00:1775145144.048717   84822 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 64253 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:06:1b.0, compute capability: 8.0


  name embeddings :   0%|          | 0/705 [00:00<?, ?batch/s]

  name embeddings  1,322,725 names embedded in 54.1s (24,444/s) | cache=miss | backend=chars2vec/tensorflow
  text embeddings  cache=miss | sources=547,517 | precomputed=0 | batch_size=auto | device=cuda | backend=transformers | precision=auto


  text embeddings :   0%|          | 0/547517 [00:00<?, ?text/s]

  text embeddings  547,517 source texts -> 1,322,725 mentions in 330s (1,661/s) | cache=miss | precomputed=0 | device=cuda
  pair inference   pair_upper_bound=35,827,436 | score_batch_size=8,192
  pair inference   cpu_stage=pair_building | cpu_workers=auto | sharding=auto | ram_budget=31GiB


  pair inference  :   0%|          | 0/35827436 [00:00<?, ?pair/s]

  pair inference   WARN Applied numeric clamping to pair scores: events=176 | cosine_non_finite=0 | cosine_below_min=0 | cosine_above_max=2652 | distance_non_finite=0 | distance_below_min=0 | distance_above_max=0
  pair inference   pair_count=35,826,651 | pairs_est=35,827,436 | workers=7 | sharding=yes | score_count=35,826,651 | device=cuda | precision=fp32 in 371s
  run              pair_score_timing read=4.0s | to_pandas=0.0s | score=15.2s | write=14.7s
  clustering       backend=sklearn_cpu | cpu_workers=auto | sharding=auto | ram_budget=31GiB


  clustering      :   0%|          | 0/205680 [00:00<?, ?block/s]

  clustering       clusters=231,348 | mentions=1,322,725 | backend=sklearn_cpu | workers=7 | sharding=yes in 194s
  export           started
  export           writing publications_disambiguated.parquet
  export           writing references_disambiguated.parquet
  export           writing 05_stage_metrics_infer_sources.json
  export           GO=True | blockers=0 in 79.6s
  done             go=True | publications=183516 | references=441957 | mentions=1322725 | clusters=231348
  summary          /home/ubuntu/Author_Name_Disambiguation/artifacts/exports/notebook_infer_integration_mwe/summary.json


In [4]:
summary = json.loads(result.summary_path.read_text(encoding='utf-8'))
summary

{'run_id': 'infer_sources_20260402T155125Z_b68359a5',
 'dataset_id': 'notebook_infer_integration_mwe',
 'output_root': '/home/ubuntu/Author_Name_Disambiguation/artifacts/exports/notebook_infer_integration_mwe',
 'summary_path': '/home/ubuntu/Author_Name_Disambiguation/artifacts/exports/notebook_infer_integration_mwe/summary.json',
 'go': True,
 'infer_stage_requested': 'full',
 'infer_stage_effective': 'full',
 'runtime_mode': 'gpu',
 'runtime_backend': 'transformers',
 'resolved_device': 'cuda',
 'precision_mode': 'amp_bf16',
 'clustering_backend': 'sklearn_cpu',
 'counts': {'publications': 183516,
  'references': 441957,
  'canonical_records': 547517,
  'specter_sources': 547517,
  'mentions': 1322725,
  'clusters': 231348,
  'authors_total': 1497609,
  'authors_mapped': 1497609,
  'authors_unmapped': 0},
 'stage_seconds': {'bootstrap': 0.672706761979498,
  'load_inputs': 38.38599414000055,
  'preflight': 16.648849490040448,
  'name_embeddings': 54.113226634974126,
  'text_embeddings

In [5]:
events_df = pd.DataFrame(
    {
        'kind': event.kind,
        'stage_key': event.stage_key,
        'stage_label': event.stage_label,
        'current': event.current,
        'total': event.total,
        'unit': event.unit,
        'message': event.message,
    }
    for event in handler.events
)
events_df.head(20)

,kind,stage_key,stage_label,current,total,unit,message
0,stage_start,bootstrap,Bootstrap,NaN,NaN,None,None
1,stage_info,bootstrap,Bootstrap,NaN,NaN,None,dataset=notebook_infer_integration_mwe | infer...
2,stage_info,bootstrap,Bootstrap,NaN,NaN,None,device=cuda -> cuda | gpu=NVIDIA A100-SXM4-80G...
3,stage_info,bootstrap,Bootstrap,NaN,NaN,None,run_id=infer_sources_20260402T155125Z_b68359a5...
4,stage_done,bootstrap,Bootstrap,NaN,NaN,None,Bootstrap in 0.7s
5,stage_start,load_inputs,Load inputs,NaN,NaN,None,None
6,stage_info,load_inputs,Load inputs,NaN,NaN,None,publications=/home/ubuntu/Author_Name_Disambig...
7,stage_done,load_inputs,Load inputs,NaN,NaN,None,"Loaded publications=183,516 references=441,957..."
8,stage_start,preflight,Preflight,NaN,NaN,None,None
9,stage_info,preflight,Preflight,NaN,NaN,None,precomputed_embedding column present=yes
